# Chapter 8: Design a URL Shortener
*System Design Interview*

## TL;DR

A URL shortener maps long URLs to 7-character short codes using **base-62 conversion** of a unique ID. Reads (redirects) vastly outnumber writes, so a **cache layer** is essential. Choose **301** redirects to reduce server load or **302** for analytics. Storage for 10 years at 100M URLs/day requires ~365 TB.

## Requirements

| Requirement | Detail |
|---|---|
| Shorten | Given a long URL, return a short URL |
| Redirect | Given a short URL, redirect to the original |
| Traffic | 100 million URLs generated per day |
| Read:Write ratio | 10:1 |
| Characters | [0-9, a-z, A-Z] = 62 possible characters |
| Lifespan | 10 years |
| High availability | Must handle failures gracefully |

## Estimation: Capacity Planning

In [ ]:
# Back-of-the-envelope estimation for URL shortener
urls_per_day = 100_000_000
seconds_per_day = 24 * 3600

write_qps = urls_per_day / seconds_per_day
read_qps = write_qps * 10  # 10:1 read/write ratio

years = 10
total_urls = urls_per_day * 365 * years
avg_url_bytes = 100
storage_bytes = total_urls * avg_url_bytes
storage_tb = storage_bytes / (1024**4)

print(f'Write QPS: {write_qps:,.0f}')
print(f'Read QPS:  {read_qps:,.0f}')
print(f'Total URLs over {years} years: {total_urls:,.0f} ({total_urls/1e9:.0f} billion)')
print(f'Storage required: {storage_tb:,.0f} TB')

## Estimation: Hash Value Length

In [ ]:
# Find minimum hash length for base-62 encoding
total_urls_needed = 365_000_000_000  # 365 billion

print(f'Need to support {total_urls_needed:,} URLs\n')
for n in range(1, 12):
    capacity = 62 ** n
    sufficient = 'YES' if capacity >= total_urls_needed else 'no'
    print(f'n={n:>2}  ->  62^{n} = {capacity:>20,}  sufficient: {sufficient}')
    if capacity >= total_urls_needed and n > 1:
        if 62 ** (n-1) < total_urls_needed:
            print(f'\n=> Minimum hash length: {n} characters')
            break

## Estimation: Base-62 Conversion Example

In [ ]:
# Demonstrate base-62 conversion
CHARS = '0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'

def to_base62(num):
    if num == 0:
        return CHARS[0]
    result = []
    while num > 0:
        result.append(CHARS[num % 62])
        num //= 62
    return ''.join(reversed(result))

def from_base62(s):
    num = 0
    for c in s:
        num = num * 62 + CHARS.index(c)
    return num

# Examples from the book
test_id = 11157
short = to_base62(test_id)
print(f'ID {test_id} -> base62: "{short}"')
print(f'base62 "{short}" -> ID: {from_base62(short)}')

# Realistic example
big_id = 2009215674938
short2 = to_base62(big_id)
print(f'\nID {big_id} -> base62: "{short2}"')
print(f'base62 "{short2}" -> ID: {from_base62(short2)}')
print(f'Short URL: https://tinyurl.com/{short2}')

## High-Level Design

```mermaid
graph TB
    subgraph Write Path
        C1[Client] -->|POST /shorten| LB1[Load Balancer]
        LB1 --> WS1[Web Server]
        WS1 --> IDGen[ID Generator]
        WS1 --> DB[(Database)]
    end

    subgraph Read Path
        C2[Client] -->|GET /shortUrl| LB2[Load Balancer]
        LB2 --> WS2[Web Server]
        WS2 --> Cache[(Cache)]
        Cache -->|miss| DB2[(Database)]
        WS2 -->|301/302| C2
    end
```

## Deep Dive

### URL Shortening Flow (Base-62)
1. Client POSTs a long URL
2. Check if long URL already exists in DB; if so, return existing short URL
3. Generate a new unique ID (via distributed ID generator)
4. Convert ID to base-62 string (7 characters)
5. Store `(id, shortURL, longURL)` in database
6. Return short URL to client

### Hash + Collision Resolution (alternative)
- Apply CRC32/MD5/SHA-1 to the long URL, take first 7 characters
- If collision, append a predefined string and re-hash
- Use **bloom filter** to speed up collision checks
- Downside: expensive DB lookups for collision detection

### URL Redirecting
- **301 Permanent**: browser caches redirect, reduces server load
- **302 Temporary**: every click hits the server, enables click analytics
- Read path: check cache first, then database

### Data Model
Simple table: `id (PK) | shortURL (indexed) | longURL`

### Scaling Considerations
- Web tier is stateless -- scale horizontally
- Database: replication + sharding
- Cache: store frequently accessed short-to-long mappings
- Rate limiting to prevent abuse

## Trade-offs

| Dimension | Option A | Option B |
|---|---|---|
| Hash + collision | No ID generator needed | Expensive collision resolution |
| Base-62 conversion | Deterministic, no collision | Requires distributed ID generator |
| 301 redirect | Less server load | Lose analytics data |
| 302 redirect | Track every click | Higher server load |
| Cache layer | Fast reads | Memory cost, cache invalidation |

## Key Takeaways

1. Base-62 conversion of a unique ID is the cleanest shortening approach
2. 7 characters of base-62 support **3.5 trillion** URLs -- plenty for 365B
3. Read-heavy workload (10:1) demands a **cache layer** for redirects
4. 301 vs 302 redirect is a product decision (performance vs analytics)
5. The distributed ID generator from Chapter 7 (Snowflake) plugs directly in

## Related Concepts

- [[url-shortening]] -- the write path and hash generation
- [[url-redirecting]] -- 301 vs 302 and the read path
- [[base62-conversion]] -- encoding numeric IDs as short strings
- [[hash-collision-resolution]] -- CRC32/MD5 approach with bloom filters
- [[back-of-envelope-estimation]] -- capacity planning calculations